In [ ]:
import pandas as pd
import numpy as np
import io
import requests
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# 1. Conversão do link de compartilhamento para link de download direto
# O ID do  arquivo é: 1u25iPqVFC3jBYunR2H8teJLgDy-pUu_B
file_id = '1u25iPqVFC3jBYunR2H8teJLgDy-pUu_B'
url = f'https://drive.google.com/uc?export=download&id={file_id}'


In [ ]:
# 2. Carregamento dos dados via requisição HTTP
response = requests.get(url)
content = response.content

In [ ]:
# Lendo o conteúdo como CSV (usando o separador ';' identificado no arquivo)
df = pd.read_csv(io.StringIO(content.decode('latin1')), sep=';')

In [ ]:
df.head()

,data_completa,ano,mes,nome_mes,dia,nome_item_despesa,valor_ordem_pagamento
0,2021-01-05,2021,1,JANEIRO,5,SERVICOS MEDICOHOSPITALARES AMBULATORIAIS,5162578.35
1,2021-01-05,2021,1,JANEIRO,5,VENCIMENTOS E SALARIOS,25.49
2,2021-01-06,2021,1,JANEIRO,6,ABONOS SALARIAIS,804992.18
3,2021-01-06,2021,1,JANEIRO,6,CONDOMINIOS,7539.00
4,2021-01-06,2021,1,JANEIRO,6,DE FERIAS CONSTITUCIONAIS,661723.38


In [ ]:
# 3. Preparação e Limpeza

df['data_completa'] = pd.to_datetime(df['ano'].astype(str) + '-' +
                                     df['mes'].astype(str) + '-' +
                                     df['dia'].astype(str), errors='coerce')


df['valor_ordem_pagamento'] = pd.to_numeric(df['valor_ordem_pagamento'], errors='coerce').fillna(0)



In [ ]:
# Agrupamento para consolidar valores por item e período
df.columns = df.columns.str.strip()
df_grouped = df.groupby(['ano', 'mes', 'nome_item_despesa'], as_index=False)['valor_ordem_pagamento'].sum()

In [ ]:
# 4. Engenharia de Variáveis
df_ml = pd.get_dummies(df_grouped, columns=['nome_item_despesa'])

In [ ]:
# 5. Divisão Treino e Teste para Validação (Jan-Fev/2026)
# O conjunto de treino agora inclui apenas dados anteriores a 2026.
train_data = df_ml[df_ml['ano'] < 2026]

# O conjunto de teste para validação inclui Jan e Fev de 2026 (onde temos valores reais).
test_actuals_data = df_ml[(df_ml['ano'] == 2026) & (df_ml['mes'] < 3)]

X_train = train_data.drop(columns=['valor_ordem_pagamento'])
y_train = train_data['valor_ordem_pagamento']

X_test_actuals = test_actuals_data.drop(columns=['valor_ordem_pagamento'])
y_test_actuals = test_actuals_data['valor_ordem_pagamento']


In [ ]:
# 6. Treinamento do Modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


RandomForestRegressor(random_state=42)

In [ ]:
# 7. Geração das Previsões e Comparativo

# --- Previsões para Jan-Fev 2026 (com dados reais para comparação) ---
# Garantir que X_test_actuals tenha as mesmas colunas que X_train e na mesma ordem
missing_cols_in_actuals = set(X_train.columns) - set(X_test_actuals.columns)
for col in missing_cols_in_actuals:
    X_test_actuals[col] = False

extra_cols_in_actuals = set(X_test_actuals.columns) - set(X_train.columns)
if extra_cols_in_actuals:
    X_test_actuals = X_test_actuals.drop(columns=list(extra_cols_in_actuals))

X_test_actuals = X_test_actuals[X_train.columns] # Reordenar colunas

previsoes_jan_feb = model.predict(X_test_actuals)

# Criar DataFrame de resultados para Jan-Fev 2026
# Precisamos dos nomes dos itens de despesa originais para mesclar com o resultado.
actuals_df_for_merge = df_grouped[(df_grouped['ano'] == 2026) & (df_grouped['mes'] < 3)].copy()
actuals_df_for_merge = actuals_df_for_merge[['ano', 'mes', 'nome_item_despesa', 'valor_ordem_pagamento']]
actuals_df_for_merge.rename(columns={'valor_ordem_pagamento': 'valor_real'}, inplace=True)

resultado_jan_feb = actuals_df_for_merge.copy()
resultado_jan_feb['valor_previsto'] = previsoes_jan_feb.round(2)
resultado_jan_feb['erro_absoluto'] = np.abs(resultado_jan_feb['valor_previsto'] - resultado_jan_feb['valor_real']).round(2)


# --- Previsões para Mar-Dez 2026 (período futuro sem dados reais) ---
# 1. Obter os nomes únicos dos itens de despesa do DataFrame agrupado
unique_items = df_grouped['nome_item_despesa'].unique()

# 2. Gerar combinações de ano, mês e item de despesa para o período de previsão (Março a Dezembro de 2026)
future_data_for_prediction = []
for mes_predicao in range(3, 13): # Meses de Março (3) a Dezembro (12)
    for item_despesa in unique_items:
        future_data_for_prediction.append({'ano': 2026, 'mes': mes_predicao, 'nome_item_despesa': item_despesa})

# 3. Criar um DataFrame a partir dessas combinações
df_future_prediction = pd.DataFrame(future_data_for_prediction)

# 4. Aplicar One-Hot Encoding à coluna 'nome_item_despesa' para criar as features para a previsão
X_test_future_generated = pd.get_dummies(df_future_prediction, columns=['nome_item_despesa'])

# 5. Garantir que X_test_future_generated tenha exatamente as mesmas colunas e ordem que X_train
missing_cols_in_future_test = set(X_train.columns) - set(X_test_future_generated.columns)
for col in missing_cols_in_future_test:
    X_test_future_generated[col] = False

extra_cols_in_future_test = set(X_test_future_generated.columns) - set(X_train.columns)
if extra_cols_in_future_test:
    X_test_future_generated = X_test_future_generated.drop(columns=list(extra_cols_in_future_test))

X_test_future = X_test_future_generated[X_train.columns] # Reordenar colunas

previsoes_mar_dec = model.predict(X_test_future)

# Criar DataFrame de resultados para Mar-Dez 2026
resultado_mar_dec = df_future_prediction.copy()
resultado_mar_dec['valor_previsto'] = previsoes_mar_dec.round(2)
resultado_mar_dec['valor_real'] = 0.00 # Preencher com 0.00 conforme solicitado
resultado_mar_dec['erro_absoluto'] = 0.00 # Preencher com 0.00 conforme solicitado

# --- Combinar os resultados ---
resultado_final = pd.concat([resultado_jan_feb, resultado_mar_dec], ignore_index=True)
resultado_final = resultado_final.sort_values(by=['ano', 'mes', 'nome_item_despesa']).reset_index(drop=True)


In [ ]:
import numpy as np
import pandas as pd

# Renomear para consistência e arredondar valores
resultado_final.rename(columns={'nome_item_despesa': 'Nome Item Despesa'}, inplace=True)
resultado_final['valor_previsto'] = resultado_final['valor_previsto'].round(2)
resultado_final['valor_real'] = resultado_final['valor_real'].round(2)
resultado_final['erro_absoluto'] = resultado_final['erro_absoluto'].round(2)

# Create a mapping for month names in Portuguese
month_names_pt = {
    1: 'JANEIRO', 2: 'FEVEREIRO', 3: 'MARÇO', 4: 'ABRIL',
    5: 'MAIO', 6: 'JUNHO', 7: 'JULHO', 8: 'AGOSTO',
    9: 'SETEMBRO', 10: 'OUTUBRO', 11: 'NOVEMBRO', 12: 'DEZEMBRO'
}

# Add 'data_completa' column (first day of the month)
resultado_final['data_completa'] = pd.to_datetime(
    resultado_final['ano'].astype(str) + '-' + resultado_final['mes'].astype(str) + '-01'
)
# Add 'Nome do Mês' column
resultado_final['Nome do Mês'] = resultado_final['mes'].map(month_names_pt)

# Calcular o percentual de acerto
resultado_final['Percentual de Acerto'] = np.where(
    resultado_final['valor_real'] > 0,
    (1 - (resultado_final['erro_absoluto'] / resultado_final['valor_real'])) * 100,
    0.00
).round(2).clip(min=0)

# Reorder columns to include 'data_completa' and 'Nome do Mês'
new_column_order = [
    'ano', 'mes', 'Nome do Mês', 'data_completa', 'Nome Item Despesa',
    'valor_real', 'valor_previsto', 'erro_absoluto', 'Percentual de Acerto'
]
resultado_final = resultado_final[new_column_order]

# Exibir as primeiras linhas do resultado final
print("Previsões para 2026 (Jan-Fev com valores reais e Erro Absoluto, Mar-Dez preenchidos com 0):")
display(resultado_final.head())

Previsões para 2026 (Jan-Fev com valores reais e Erro Absoluto, Mar-Dez preenchidos com 0):


,ano,mes,Nome do Mês,data_completa,Nome Item Despesa,valor_real,valor_previsto,erro_absoluto,Percentual de Acerto
0,2026,1,JANEIRO,2026-01-01,ABONOS SALARIAIS,353148.40,744772.88,391624.48,0.00
1,2026,1,JANEIRO,2026-01-01,ARMAZENAGEM,1335.13,18815.80,17480.67,0.00
2,2026,1,JANEIRO,2026-01-01,ARQUIVO DE DOCUMENTOS,2418.28,2347.34,70.94,97.07
3,2026,1,JANEIRO,2026-01-01,COMISSOES E CORRETAGENS,29700.00,81498.86,51798.86,0.00
4,2026,1,JANEIRO,2026-01-01,CONDOMINIOS,409257.56,398216.96,11040.60,97.30


In [ ]:
# Re-exportando o arquivo CSV com a nova coluna de 'Percentual de Acerto'
output_name = 'previsao_despesas_compartilhada_2026.csv'
resultado_final.to_csv(output_name, index=False, sep=';', encoding='utf-8-sig')

print(f"Novo arquivo '{output_name}' gerado com sucesso, incluindo o percentual de acerto.")

Novo arquivo 'previsao_despesas_compartilhada_2026.csv' gerado com sucesso, incluindo o percentual de acerto.


In [ ]:
# Filtrar itens com percentual de acerto maior ou igual a 90%
df_filtered_accuracy = resultado_final[resultado_final['Percentual de Acerto'] >= 90].copy()

# Definir o nome do novo arquivo de saída
output_filename_high_accuracy = 'previsao_despesas_alta_acuracia_2026.csv'

# Exportar o DataFrame filtrado para um novo arquivo CSV
df_filtered_accuracy.to_csv(output_filename_high_accuracy, index=False, sep=';', encoding='utf-8-sig')

print(f"Novo arquivo '{output_filename_high_accuracy}' gerado com sucesso, contendo itens com percentual de acerto >= 90%.")

# Exibir as primeiras linhas do DataFrame filtrado
print("Itens com Percentual de Acerto >= 90% (incluindo as novas colunas):")
display(df_filtered_accuracy.head())

Novo arquivo 'previsao_despesas_alta_acuracia_2026.csv' gerado com sucesso, contendo itens com percentual de acerto >= 90%.
Itens com Percentual de Acerto >= 90% (incluindo as novas colunas):


,ano,mes,Nome do Mês,data_completa,Nome Item Despesa,valor_real,valor_previsto,erro_absoluto,Percentual de Acerto
2,2026,1,JANEIRO,2026-01-01,ARQUIVO DE DOCUMENTOS,2418.28,2347.34,70.94,97.07
4,2026,1,JANEIRO,2026-01-01,CONDOMINIOS,409257.56,398216.96,11040.60,97.30
7,2026,1,JANEIRO,2026-01-01,DESENVOLVIMENTO DE SOFWARE,37304861.83,40821045.18,3516183.35,90.57
9,2026,1,JANEIRO,2026-01-01,DIARIAS DE DEMAIS SERVIDORES,758480.31,784782.00,26301.69,96.53
19,2026,1,JANEIRO,2026-01-01,LOCACAO DE MAQUINAS E EQUIPAMENTOS,363568.66,379649.79,16081.13,95.58


In [ ]:
print("Primeiras linhas de 'resultado_final' com as novas colunas:")
display(resultado_final.head())

Primeiras linhas de 'resultado_final' com as novas colunas:


,ano,mes,Nome do Mês,data_completa,Nome Item Despesa,valor_real,valor_previsto,erro_absoluto,Percentual de Acerto
0,2026,1,JANEIRO,2026-01-01,ABONOS SALARIAIS,353148.40,744772.88,391624.48,0.00
1,2026,1,JANEIRO,2026-01-01,ARMAZENAGEM,1335.13,18815.80,17480.67,0.00
2,2026,1,JANEIRO,2026-01-01,ARQUIVO DE DOCUMENTOS,2418.28,2347.34,70.94,97.07
3,2026,1,JANEIRO,2026-01-01,COMISSOES E CORRETAGENS,29700.00,81498.86,51798.86,0.00
4,2026,1,JANEIRO,2026-01-01,CONDOMINIOS,409257.56,398216.96,11040.60,97.30


In [ ]:
# 8. Exportação
# Nota: O arquivo final ainda será salvo no Drive de quem executar ou no ambiente local do Colab
output_name = 'previsao_despesas_compartilhada_2026.csv' # Nome do arquivo alterado para refletir o novo escopo
resultado_final.to_csv(output_name, index=False, sep=';', encoding='utf-8-sig')

print(f"Previsão finalizada para {len(resultado_final)} itens. Arquivo '{output_name}' gerado.")


Previsão finalizada para 1014 itens. Arquivo 'previsao_despesas_compartilhada_2026.csv' gerado.
